# 7.2 Python Packages

**Prerequisites:** 7.1 Python Module  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a package is: `__init__.py`, subpackages, namespace packages
- Absolute vs relative imports, and why relative imports break in scripts
- `__all__` and controlling a package's public surface
- **`python -m venv`** — virtual environments, the built-in way
- `pip` properly: install, list, freeze, version specifiers, editable installs
- `requirements.txt`, pinning, and why it matters

---

### Libraries/Packages in Python:
- Python Package Index (PyPI) is a repository of software for the Python programming language. 
    - PyPI helps us to find and install software developed and shared by the Python community.
    - Also Package authors use PyPI to distribute their software.
- Libraries in python are called packages. Packages encapsulate code in file called modules.
- A package, in essence, is like a directory holding subpackages and modules that can be easily imported for use in python program.
    - The directory must have a `__init__.py` file to be called as a python package.
- A package can contain namespaces, modules(.py files) and nested packages.
- The packages in python facilitate the developer with the application development environment by providing a hierarchical directory structure.
    - The packages are used to categorize the application level code efficiently.
- While we can use the packages from the Python Package Index (PyPI), we can also create our own packages.


### Ways of installing packages from repository:
- Two common ways of installing additional packages are:
    - pip
    - conda
<img src='./Image/7.1 Image a.png'>

> **Note on this notebook.** The original version imported from a `Sample_Package/` folder
> that no longer exists, so those cells could not run. Everything below now **builds a real
> package in a temporary directory at run time**, imports from it, and cleans up — so the
> notebook is self-contained and cannot rot.

---

## 1. Module vs package

| | Module | Package |
|---|---|---|
| **Is** | A single `.py` file | A **directory** of modules |
| **Import** | `import utils` | `import myapp.utils` |
| **Marked by** | The `.py` extension | Usually an `__init__.py` file |

```
myapp/                     <- the package
├── __init__.py            <- runs when `myapp` is imported
├── config.py              <- module: myapp.config
├── models.py              <- module: myapp.models
└── storage/               <- SUBpackage: myapp.storage
    ├── __init__.py
    ├── local.py           <- myapp.storage.local
    └── s3.py              <- myapp.storage.s3
```

### What `__init__.py` is actually for

Three jobs, in order of how often you'll use them:

1. **Marking the directory as a package** — historically its only job.
2. **Defining the package's public API** — re-export the names users should reach for, so
   they write `from myapp import Client` instead of `from myapp.client.base import Client`.
3. **Package-level setup** — rare, and usually a mistake: it runs on *every* import, so slow
   or side-effecting code here makes importing your package slow for everyone.

> **Version note — namespace packages.** Since **Python 3.3** a directory **without**
> `__init__.py` can still be imported (PEP 420). This is convenient but usually accidental:
> a missing `__init__.py` used to be a loud error and is now silent. **Include
> `__init__.py`** unless you are deliberately building a namespace package split across
> distributions.

In [ ]:
import shutil, sys, tempfile, textwrap
from pathlib import Path

# ---- Build a real package in a temp directory ----
root = Path(tempfile.mkdtemp(prefix="pkg_demo_"))
pkg = root / "myapp"
(pkg / "storage").mkdir(parents=True)


def write(path: Path, text: str) -> None:
    path.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")


write(pkg / "__init__.py", """
    'myapp - a demonstration package.'

    __version__ = "1.0.0"

    # Re-export the public API so callers write `from myapp import Client`
    from myapp.client import Client
    from myapp.config import Settings

    __all__ = ["Client", "Settings", "__version__"]

    print("  [myapp/__init__.py executed]")
""")

write(pkg / "config.py", """
    'Configuration objects.'
    from dataclasses import dataclass

    @dataclass(frozen=True)
    class Settings:
        base_url: str = "https://api.example.com"
        timeout: float = 30.0
""")

write(pkg / "client.py", """
    'The public client.'
    from myapp.config import Settings          # ABSOLUTE import

    class Client:
        def __init__(self, settings=None):
            self.settings = settings or Settings()

        def __repr__(self):
            return f"Client(base_url={self.settings.base_url!r})"
""")

write(pkg / "storage" / "__init__.py", """
    'Storage backends.'
    from myapp.storage.local import LocalStorage

    __all__ = ["LocalStorage"]
""")

write(pkg / "storage" / "local.py", """
    'Local-filesystem storage.'
    from ..config import Settings              # RELATIVE import (.. = myapp)

    class LocalStorage:
        def __init__(self):
            self.settings = Settings()

        def describe(self):
            return f"LocalStorage using {self.settings.base_url}"

    if __name__ == "__main__":
        print(LocalStorage().describe())
""")

# Show the tree we just built
print("package tree:")
for p in sorted(root.rglob("*")):
    depth = len(p.relative_to(root).parts) - 1
    print("  " + "    " * depth + p.name + ("/" if p.is_dir() else ""))

sys.path.insert(0, str(root))

In [ ]:
# ---- Importing it ----
print("importing myapp:")
import myapp

print("\nversion   :", myapp.__version__)
print("__all__   :", myapp.__all__)

# Because __init__.py re-exported them, both of these work:
client = myapp.Client()
print("\nfrom the package root :", client)

from myapp.client import Client as DeepClient
print("from the deep module  :", DeepClient())

# Subpackage
from myapp.storage import LocalStorage
print("\nsubpackage            :", LocalStorage().describe())

# A package is a module object whose __path__ lists its directories
print("\nmyapp is a", type(myapp).__name__)
print("myapp.__path__ ends with:", Path(myapp.__path__[0]).name)

# Everything that got imported
loaded = sorted(m for m in sys.modules if m.startswith("myapp"))
print("\nmodules now in sys.modules:")
for m in loaded:
    print("  ", m)

---

## 2. Absolute vs relative imports

Inside a package you can refer to a sibling module two ways:

```python
from myapp.config import Settings      # ABSOLUTE - full path from the package root
from .config import Settings           # RELATIVE - . means "this package"
from ..config import Settings          # .. means "the parent package"
```

| | Absolute | Relative |
|---|---|---|
| Readability | Explicit — you can see exactly what is imported | Shorter, and survives renaming the package |
| Works when the file is **run directly** | ✅ | ❌ **`ImportError`** |
| PEP 8 says | **Preferred** | Acceptable for intra-package imports |

### ⚠️ Why relative imports fail when you run a file directly

```bash
python myapp/storage/local.py
```

Run that way, `local.py` has `__name__ == "__main__"` and **no package context**, so `..` has
nothing to resolve against. You get:

```
ImportError: attempted relative import with no known parent package
```

The fix is to run it **as a module**, which gives Python the package context:

```bash
python -m myapp.storage.local
```

This is one of the most common confusions in Python packaging, and the error message is not
obvious about the cause.

In [ ]:
import subprocess

script = pkg / "storage" / "local.py"

# ---- Running the file directly: the relative import fails ----
result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True, text=True, cwd=str(root),
)
print("python myapp/storage/local.py")
print("  ->", (result.stderr.strip().splitlines() or ["(no error)"])[-1])

# ---- Running it as a module: works, because the package context exists ----
result = subprocess.run(
    [sys.executable, "-m", "myapp.storage.local"],
    capture_output=True, text=True, cwd=str(root),
)
status = "ok" if result.returncode == 0 else result.stderr.strip().splitlines()[-1]
print("\npython -m myapp.storage.local")
print("  ->", status)

print("""
Rule of thumb:
  - Use ABSOLUTE imports (from myapp.config import Settings) by default.
  - Use `python -m package.module`, not `python path/to/file.py`, to run
    anything that lives inside a package.
""")


# ---- Clean up the demo package ----
for name in [m for m in sys.modules if m.startswith("myapp")]:
    del sys.modules[name]
sys.path.remove(str(root))
shutil.rmtree(root)
print("demo package removed:", not root.exists())

---

## 3. Virtual environments

> ### 🔴 Correction to the original notes
> The original version of this notebook said to run `pip3 install virtualenv` and then
> `virtualenv pythonenv -p python3`.
>
> **`venv` has been part of the standard library since Python 3.3.** You do not need to
> install anything. The original also gave `ls` as the way to "see the list of packages
> installed" — `ls` lists *files*; the command is **`pip list`**.

### The problem venv solves

Install packages globally and every project shares them. Project A needs Django 4, project B
needs Django 5 — one of them is broken, permanently. A **virtual environment** is a private
directory holding its own Python interpreter and its own `site-packages`.

**One environment per project.** Always.

### The commands

```bash
# Create - the folder name is conventionally .venv
python -m venv .venv

# Activate
.venv\Scripts\activate           # Windows (cmd)
.venv\Scripts\Activate.ps1       # Windows (PowerShell)
source .venv/bin/activate         # macOS / Linux

# Confirm you are inside it
python -c "import sys; print(sys.prefix)"
pip list

# Leave
deactivate
```

Once activated, `python` and `pip` refer to the environment's copies — so `pip install`
touches only this project.

> **Add `.venv/` to `.gitignore`.** The environment is rebuilt from `requirements.txt`; it is
> not source code and must never be committed.

### How to tell whether you are in one

In [ ]:
import sys, os
from pathlib import Path

print("sys.executable :", sys.executable)
print("sys.prefix     :", sys.prefix)
print("base_prefix    :", sys.base_prefix)

# The canonical check: in a venv, prefix and base_prefix differ
in_venv = sys.prefix != sys.base_prefix
print("\nrunning inside a virtual environment:", in_venv)

# VIRTUAL_ENV is set by the activation script
print("VIRTUAL_ENV env var:", os.environ.get("VIRTUAL_ENV", "(not set)"))

# Where packages are installed for THIS interpreter
import site
print("\nsite-packages:")
for p in site.getsitepackages()[:2]:
    print("  ", p)

print("""
If in_venv is False you are using a global or system Python. That is fine for
this notebook, but for a real project create one:

    python -m venv .venv
""")

---

## 4. `pip`

`pip` installs packages from **PyPI** (the Python Package Index) into whichever environment
is currently active.

### The commands worth knowing

| Command | Does |
|---|---|
| `pip install requests` | Install the latest version |
| `pip install "requests==2.31.0"` | Install exactly that version |
| `pip install "requests>=2.28,<3"` | Install within a range |
| `pip install -r requirements.txt` | Install everything listed in a file |
| `pip install -e .` | **Editable install** of the local project |
| `pip uninstall requests` | Remove it |
| `pip list` | What is installed (human-readable) |
| `pip freeze` | What is installed (machine-readable, exact versions) |
| `pip show requests` | Version, location, dependencies |
| `pip install --upgrade requests` | Upgrade |

> **Always run it as `python -m pip install ...`** rather than bare `pip`. That guarantees you
> are installing into *the interpreter you think you are*, which matters the moment you have
> more than one Python on the machine.

### `pip list` vs `pip freeze`

- **`pip list`** — a readable table, includes pip and setuptools. For humans.
- **`pip freeze`** — `name==version` lines, ready to redirect into `requirements.txt`. For machines.

### Version specifiers

| Specifier | Means |
|---|---|
| `requests` | Any version — ⚠️ not reproducible |
| `requests==2.31.0` | Exactly this |
| `requests>=2.28` | This or newer |
| `requests>=2.28,<3.0` | Compatible range — the usual choice for libraries |
| `requests~=2.31.0` | "Compatible release": `>=2.31.0, <2.32.0` |

### Editable installs

`pip install -e .` installs your project as a **link** to the source directory rather than a
copy. Edit the code and the change is live — no reinstall.

This is the correct answer to "how do I import my package from a test file two directories
away", and the reason you should not be mutating `sys.path` (as **7.1** demonstrates).

In [ ]:
import subprocess, sys

def pip(*args: str) -> str:
    """Run pip against THIS interpreter and return its output."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args],
        capture_output=True, text=True,
    )
    return (result.stdout or result.stderr).strip()


print("pip version:", pip("--version").split(" from ")[0])

# ---- pip list: human-readable ----
listing = pip("list").splitlines()
print(f"\npip list ({len(listing) - 2} packages), first few:")
for line in listing[:6]:
    print("  ", line)

# ---- pip freeze: machine-readable, exact pins ----
frozen = [line for line in pip("freeze").splitlines() if line and not line.startswith("-")]
print(f"\npip freeze ({len(frozen)} lines), first few:")
for line in frozen[:5]:
    print("  ", line)

print("""
  ^ notice the difference: `list` is a table for you to read,
    `freeze` is `name==version` lines ready for requirements.txt.
""")

# ---- pip show: details of one package ----
info = pip("show", "pip")
for line in info.splitlines():
    if line.split(":")[0] in {"Name", "Version", "Location", "Requires"}:
        print("  ", line)

---

## 5. `requirements.txt`

A plain text file listing what your project needs, one per line.

```
# requirements.txt
requests>=2.28,<3.0
python-dateutil~=2.8.2
rich==13.7.0
```

```bash
python -m pip freeze > requirements.txt        # capture what you have
python -m pip install -r requirements.txt      # recreate it elsewhere
```

### Pin, or don't pin?

This is a real decision, and the answer depends on what you are building:

| You are building | Pin how | Why |
|---|---|---|
| An **application** (a service, a script, a job) | **Exact** — `==2.31.0` | Reproducible deploys; you control the upgrade |
| A **library** others install | **Ranges** — `>=2.28,<3.0` | Over-pinning makes your library impossible to co-install |

### The convention

Two files, for the two audiences:

```
requirements.txt        # what the app needs to run
requirements-dev.txt    # plus pytest, mypy, ruff...
```

> **Where this is going.** `requirements.txt` is the traditional approach. The modern one is
> **`pyproject.toml`**, which declares dependencies *and* project metadata *and* tool
> configuration in one file, plus a **lock file** that pins the entire transitive tree.
> Tools: `pip`, `uv`, `poetry`, `pdm`. That is **17 Tooling, Packaging and Environments**.

In [ ]:
import tempfile, textwrap
from pathlib import Path

demo = Path(tempfile.mkdtemp(prefix="reqs_demo_"))

# ---- A typical requirements.txt ----
reqs = demo / "requirements.txt"
reqs.write_text(textwrap.dedent("""
    # Core dependencies
    requests>=2.28,<3.0
    python-dateutil~=2.8.2

    # Pinned because 13.8 changed the table API
    rich==13.7.0
""").lstrip(), encoding="utf-8")

dev_reqs = demo / "requirements-dev.txt"
dev_reqs.write_text(textwrap.dedent("""
    -r requirements.txt

    pytest>=8.0
    mypy>=1.8
    ruff>=0.3
""").lstrip(), encoding="utf-8")

for f in (reqs, dev_reqs):
    print(f"--- {f.name} ---")
    print(f.read_text(encoding="utf-8"))


# ---- Parsing one, to see what the specifiers mean ----
import re

print("--- parsed ---")
for line in reqs.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    match = re.match(r"^([A-Za-z0-9_.\-]+)(.*)$", line)
    name, spec = match.group(1), match.group(2) or "(any version)"
    print(f"  {name:<18} {spec}")

import shutil
shutil.rmtree(demo)
print("\ncleaned up")

---

## Common Mistakes & Pitfalls

1. 🔴 **Installing `virtualenv`.** `python -m venv` is built in since 3.3.
2. **Committing `.venv/` to git.** It is large, machine-specific, and rebuildable. Add it to `.gitignore`.
3. **Installing packages globally.** One environment per project, always.
4. **Running bare `pip` when several Pythons are installed.** Use `python -m pip` so you install into the interpreter you mean.
5. **Forgetting to activate** the environment, then wondering why an import fails.
6. **Running a file inside a package directly** (`python myapp/mod.py`) and hitting `attempted relative import with no known parent package`. Use `python -m myapp.mod`.
7. **Forgetting `__init__.py`.** Since 3.3 it silently becomes a namespace package instead of erroring, which produces confusing import behaviour later.
8. **Heavy code in `__init__.py`.** It runs on every import and makes your package slow for everyone.
9. **`pip freeze > requirements.txt` from a global environment.** You capture every package on the machine, not your project's.

## Best Practices

- **One virtual environment per project**, named `.venv`, gitignored.
- Create with `python -m venv .venv`; install with `python -m pip`.
- Include `__init__.py` in every package directory unless you deliberately want a namespace package.
- Use `__init__.py` to re-export your public API, and `__all__` to declare it.
- Prefer **absolute imports**; use relative ones only within a package.
- Run package code with `python -m package.module`.
- Use `pip install -e .` for local development instead of mutating `sys.path`.
- **Pin exactly** for applications, **use ranges** for libraries.
- Keep runtime and development dependencies in separate files.

## Practice Exercises

Try these before moving on.

1. Create a `.venv`, activate it, install `requests`, and confirm with `pip list` and `sys.prefix` that it is isolated.
2. Build a package `shop/` with `__init__.py`, `models.py` and a `payments/` subpackage. Import from all three levels.
3. Add `__all__` to your package's `__init__.py` and show what `from shop import *` gives.
4. Reproduce `attempted relative import with no known parent package`, then fix it with `python -m`.
5. Delete an `__init__.py` and observe that the import still works. Explain why that is not good news.
6. Write a `requirements.txt` with one exact pin, one range and one `~=`, and explain each choice.
7. Run `pip install -e .` on a small project and confirm that editing the source takes effect without reinstalling.
8. Compare `pip list` and `pip freeze` output. Why are there two commands?